# Drum hits to pads

Nov 1. Could drum hits from a clip be handed to a keyboard's pads?

**Data:** AlbumDB close-mic stems. **What it decided:** kick is usable (onset F1 0.76 through Demucs) and snare is limited by the method. Measured, not built.

This is a selection of the cells from that work, with the output they printed. They rely on earlier cells and on a Drive folder of datasets and outputs that is not in this repository.


## Drum hits


In [ ]:
"""T1: can Mixray hand kick and snare hits to the Launchkey's pads? Measured on real close-mic stems.

Nine AlbumDB songs record kick and snare on their own microphones, so their hits are known. Each is
mixed with vocals, bass, guitar and the overheads, separated by htdemucs_6s, and the drums stem is
split by frequency band into kick and snare hits. Two controls: the same method on the raw mixture
(does Demucs help?) and on the clean drum sum (the method's own ceiling).
"""

DRUMS = r'''
import json
import subprocess
import sys
import tempfile
from pathlib import Path

import librosa
import mir_eval
import numpy as np
import soundfile as sf

RATE = 48000
START, SECONDS = 60.0, 30.0
HOP = 256
BANDS = {"kick": (30.0, 150.0), "snare": (180.0, 4000.0)}


def read(path):
    info = sf.info(str(path))
    audio, rate = sf.read(str(path), dtype="float64", always_2d=True,
                          start=int(START * info.samplerate), frames=int(SECONDS * info.samplerate))
    x = audio.mean(axis=1)
    if rate != RATE:
        x = librosa.resample(x, orig_sr=rate, target_sr=RATE)
    return np.asarray(x, dtype=np.float64)


def onsets(signal, band, delta):
    """Spectral-flux onsets restricted to one band: the kick lives low, the snare above it."""
    low, high = band
    mel = librosa.feature.melspectrogram(y=signal, sr=RATE, hop_length=HOP, n_mels=32,
                                         fmin=low, fmax=high)
    strength = librosa.onset.onset_strength(S=librosa.power_to_db(mel), sr=RATE, hop_length=HOP)
    frames = librosa.onset.onset_detect(onset_envelope=strength, sr=RATE, hop_length=HOP,
                                        delta=delta, backtrack=False, units="frames")
    return librosa.frames_to_time(frames, sr=RATE, hop_length=HOP)


def f1(reference, estimate):
    if len(reference) == 0 or len(estimate) == 0:
        return 0.0
    value, _, _ = mir_eval.onset.f_measure(np.asarray(reference), np.asarray(estimate), window=0.05)
    return float(value)


def main():
    songs_root, out_root, demucs_python = (Path(a) for a in sys.argv[1:4])
    rows = []
    for song in sorted(songs_root.iterdir()):
        names = sorted(p.name for p in song.glob("*.wav"))
        kick = next((n for n in names if n.startswith("drums_kick")), None)
        snare = next((n for n in names if n.startswith("drums_snare")), None)
        if not kick or not snare or "drums_overheads.wav" not in names:
            continue
        others = [n for n in ("vox_lead.wav", "bass.wav") if n in names]
        others += [next(n for n in names if n.startswith("guitar"))]
        stems = {n: read(song / n) for n in [kick, snare, "drums_overheads.wav", *others]}
        drums_clean = stems[kick] + stems[snare] + stems["drums_overheads.wav"]
        mixture = sum(stems.values())
        mixture = mixture * (0.891 / float(np.max(np.abs(mixture))))
        case = out_root / song.name.split(" ")[0]
        case.mkdir(parents=True, exist_ok=True)
        sf.write(str(case / "mix.wav"), mixture.astype(np.float32), RATE, subtype="PCM_16")
        subprocess.run([str(demucs_python), "-m", "demucs.separate", "-n", "htdemucs_6s",
                        "--device", "cpu", "--out", str(case / "raw"), "--filename",
                        "{stem}.{ext}", str(case / "mix.wav")], check=True, capture_output=True)
        served = read_full(next((case / "raw").rglob("drums.wav")), len(mixture))
        truth = {"kick": onsets(stems[kick], (30.0, 20000.0), 0.30),
                 "snare": onsets(stems[snare], (150.0, 20000.0), 0.30)}
        row = {"song": song.name, "truth_hits": {k: len(v) for k, v in truth.items()}}
        for part, band in BANDS.items():
            row[part] = {source: f1(truth[part], onsets(signal, band, 0.20))
                         for source, signal in (("mixture", mixture), ("demucs", served),
                                                ("clean drums", drums_clean))}
        rows.append(row)
        print(f"  {song.name:32s} kick {row['kick']}  snare {row['snare']}", flush=True)
    Path("/content/drums.json").write_text(json.dumps(rows, indent=2))
    print("DRUMS_INNER_DONE")


def read_full(path, length):
    audio, rate = sf.read(str(path), dtype="float64", always_2d=True)
    x = audio.mean(axis=1)
    if rate != RATE:
        x = librosa.resample(x, orig_sr=rate, target_sr=RATE)
    return np.asarray(x, dtype=np.float64)[:length]


if __name__ == "__main__":
    main()
'''
Path("/content/drums.py").write_text(DRUMS)
lines = ["set -o pipefail", "echo DRUMS_START",
         f"CUDA_VISIBLE_DEVICES=-1 {UV} run --no-sync --project {PROJECTS['pitch']} python "
         f"/content/drums.py {SONGS} /content/drums {demucs_python} || exit 1",
         "echo DRUMS_DONE"]
(WORK / "drums.sh").write_text("\n".join(lines))
JOB_LOG, JOB_PID, JOB_DONE_MARKER = "/content/drums.log", "/content/drums.pid", "DRUMS_DONE"
with open(JOB_LOG, "w") as handle:
    job = subprocess.Popen(["bash", str(WORK / "drums.sh")], env=ENV, stdout=handle,
                           stderr=subprocess.STDOUT, start_new_session=True)
Path(JOB_PID).write_text(str(job.pid))
print(f"drum hit test over nine songs as pid {job.pid}")


drum hit test over nine songs as pid 87741


In [ ]:
"""T1 results with the hit counts visible: an F1 of 0.0 or 0.986 means little without them."""

from statistics import mean

drums = json.loads(Path("/content/drums.json").read_text())
print(f"  {'song':30s} {'kick hits':>9s} {'snare hits':>10s}")
for row in drums:
    print(f"  {row['song']:30s} {row['truth_hits']['kick']:9d} {row['truth_hits']['snare']:10d}")
for part in ("kick", "snare"):
    usable = [r for r in drums if r["truth_hits"][part] >= 10]
    print(f"\n{part}: {len(usable)}/{len(drums)} songs have at least 10 reference hits in the window")
    for source in ("mixture", "demucs", "clean drums"):
        print(f"  {source:12s} F1 {mean(r[part][source] for r in usable):.3f}")
size = sum(p.stat().st_size for p in parts if p.exists()) / 2**30


  song                           kick hits snare hits
  01 Extraordinary                      38         21
  02 No More Mr Nice Guy                48         44
  03 Karma                              28         11
  04 Magistrates                        29         21
  05 Diva                               34         21
  06 Let The Light In                   63          0
  07 Take On The Seas                   70          0
  08 An Evening To Myself               40         29
  09 Great British Summer               49         30

kick: 9/9 songs have at least 10 reference hits in the window
  mixture      F1 0.541
  demucs       F1 0.759
  clean drums  F1 0.676

snare: 7/9 songs have at least 10 reference hits in the window
  mixture      F1 0.532
  demucs       F1 0.505
  clean drums  F1 0.547

